# LangChain Agentic RAG — Vietnamese Legal Documents

A LangGraph-based agentic RAG built on top of this repo's existing retrieval stack.

The agent follows the pattern from `examples/langgraph_agentic_rag.ipynb` and extends it for the Vietnamese legal domain:

```
START
  └─► agent (LLM selects retrieval tool)
        ├─► [no tool call] → END
        └─► retrieve (ToolNode: hybrid or graph)
              └─► grade_documents
                    ├─► [relevant] → generate → END
                    └─► [not relevant] → rewrite → agent
```

**Two retrieval tools:**
- `legal_hybrid_search` — BM25 + dense RRF fusion (factual / temporal / reasoning queries)
- `legal_graph_search`  — multi-hop graph traversal (amendment / reference chains)

## 1. Setup

In [ ]:
import sys
sys.path.insert(0, "..")

from dotenv import load_dotenv
load_dotenv("../.env")

## 2. Retrieval Tools

Wrap the repo's existing retrieval functions as LangChain `@tool`s.
The agent will choose between them based on query type.

In [ ]:
from langchain_core.tools import tool
from src.tools.retrieval_tools import _get_bm25, _get_graph, _get_store


def _format_docs(docs) -> str:
    """Format a list of Documents into a readable string with citation markers."""
    if not docs:
        return "Không tìm thấy tài liệu liên quan."
    parts = []
    for i, doc in enumerate(docs, 1):
        doc_id = doc.metadata.get("doc_id", "unknown")
        title  = doc.metadata.get("title", "")
        parts.append(f"[{i}] doc_id={doc_id}  {title}\n{doc.page_content}")
    return "\n\n---\n\n".join(parts)


@tool
def legal_hybrid_search(query: str, k: int = 5) -> str:
    """
    Search Vietnamese legal documents with hybrid BM25 + dense retrieval (RRF fusion).

    Use this tool for:
    - Factual questions: specific rules, definitions, deadlines, penalties
    - Temporal questions: regulations in effect after/before a date
    - Reasoning questions: applying rules to a scenario

    Args:
        query: Search query in Vietnamese.
        k: Number of documents to return.
    """
    from src.retrieval.hybrid import hybrid_search
    docs = hybrid_search(_get_store(), _get_bm25(), query, k=k)
    return _format_docs(docs)


@tool
def legal_graph_search(query: str, k: int = 5) -> str:
    """
    Search Vietnamese legal documents by traversing the relationship graph.
    Seeds with dense search, then follows amendment / reference / replacement edges.

    Use this tool for:
    - Multi-hop questions: what amended X, what replaced Y, documents referencing Z
    - Keywords: sửa đổi, thay thế, bãi bỏ, tham chiếu, được quy định tại

    Args:
        query: Search query in Vietnamese.
        k: Number of documents to return.
    """
    from src.retrieval.graph import graph_search
    docs = graph_search(_get_store(), _get_graph(), query, k=k)
    return _format_docs(docs)


tools = [legal_hybrid_search, legal_graph_search]
print("Tools registered:", [t.name for t in tools])

## 3. Agent State

State is a list of messages — each node appends to it.

In [ ]:
from typing import Annotated, Sequence, TypedDict
from langchain_core.messages import BaseMessage
from langgraph.graph.message import add_messages


class AgentState(TypedDict):
    messages: Annotated[Sequence[BaseMessage], add_messages]

## 4. Graph Nodes

Four nodes mirroring the example notebook:
- **`agent`** — LLM with tools, decides which retrieval tool to call (or ends)
- **`grade_documents`** — conditional edge that checks if retrieved docs are relevant
- **`generate`** — produces the final cited answer in Vietnamese
- **`rewrite`** — rewrites the query when docs aren't relevant, retries via agent

In [ ]:
from typing import Literal

from langchain_core.messages import HumanMessage, SystemMessage
from langchain_core.output_parsers import StrOutputParser
from langchain_core.prompts import ChatPromptTemplate, PromptTemplate
from pydantic import BaseModel, Field
from langgraph.prebuilt import tools_condition

from src.llm import get_llm


# ── System prompt that guides tool selection ──────────────────────────────────
AGENT_SYSTEM = SystemMessage(content="""\
Bạn là trợ lý pháp lý chuyên về văn bản pháp luật Việt Nam.

Hãy sử dụng công cụ phù hợp để tìm kiếm tài liệu trước khi trả lời:

• legal_graph_search  — dùng khi câu hỏi liên quan đến:
  sửa đổi, thay thế, bãi bỏ, tham chiếu, "được quy định tại"

• legal_hybrid_search — dùng cho tất cả các câu hỏi còn lại:
  sự kiện cụ thể, định nghĩa, điều kiện, quy trình, thời hạn,
  quy định còn/hết hiệu lực, áp dụng pháp luật vào tình huống

Nếu câu trả lời đã đủ dựa trên ngữ cảnh hiện có, không cần gọi thêm công cụ.
""")


# ── Node: agent ───────────────────────────────────────────────────────────────
def agent(state: AgentState) -> dict:
    """LLM with tools decides which retrieval tool to call, or generates directly."""
    print("---AGENT---")
    llm = get_llm().bind_tools(tools)
    messages = [AGENT_SYSTEM] + list(state["messages"])
    response = llm.invoke(messages)
    return {"messages": [response]}


# ── Edge: grade_documents ────────────────────────────────────────────────────
class GradeResult(BaseModel):
    """Binary relevance score for a retrieved document."""
    binary_score: Literal["yes", "no"] = Field(
        description="'yes' if the document contains relevant legal information, 'no' otherwise"
    )


def grade_documents(state: AgentState) -> Literal["generate", "rewrite"]:
    """Assess whether the retrieved documents are relevant to the original question."""
    print("---GRADE DOCUMENTS---")
    messages = state["messages"]
    question = messages[0].content
    retrieved_context = messages[-1].content  # last ToolMessage

    prompt = PromptTemplate.from_template(
        "Bạn là người đánh giá mức độ liên quan của tài liệu pháp lý với câu hỏi.\n\n"
        "Câu hỏi: {question}\n\n"
        "Tài liệu được truy xuất:\n{context}\n\n"
        "Nếu tài liệu chứa thông tin giúp trả lời câu hỏi, trả lời 'yes'. "
        "Nếu không liên quan, trả lời 'no'."
    )
    grader = get_llm().with_structured_output(GradeResult)
    result = (prompt | grader).invoke({
        "question": question,
        "context": retrieved_context[:3000],
    })

    if result.binary_score == "yes":
        print("---DECISION: RELEVANT → generate---")
        return "generate"
    print("---DECISION: NOT RELEVANT → rewrite---")
    return "rewrite"


# ── Node: generate ────────────────────────────────────────────────────────────
GENERATE_PROMPT = ChatPromptTemplate.from_messages([
    ("system",
     "Bạn là trợ lý pháp lý chuyên về văn bản pháp luật Việt Nam.\n"
     "Trả lời câu hỏi dựa trên các tài liệu được cung cấp.\n"
     "Trích dẫn số hiệu văn bản (doc_id) cụ thể khi có thể.\n"
     "Nếu tài liệu không đủ thông tin, hãy nói rõ điều đó."),
    ("human", "Tài liệu tham khảo:\n\n{context}\n\nCâu hỏi: {question}"),
])


def generate(state: AgentState) -> dict:
    """Generate a cited answer in Vietnamese from retrieved documents."""
    print("---GENERATE---")
    messages = state["messages"]
    question = messages[0].content
    context  = messages[-1].content

    chain = GENERATE_PROMPT | get_llm() | StrOutputParser()
    answer = chain.invoke({"question": question, "context": context})
    return {"messages": [answer]}


# ── Node: rewrite ─────────────────────────────────────────────────────────────
def rewrite(state: AgentState) -> dict:
    """Rewrite the query using synonyms or simplification, then retry retrieval."""
    print("---REWRITE QUERY---")
    question = state["messages"][0].content
    rewrite_prompt = HumanMessage(content=(
        f"Câu hỏi gốc:\n{question}\n\n"
        "Hãy viết lại câu hỏi này theo cách khác (dùng từ đồng nghĩa, đơn giản hóa,"
        " hoặc tách thành câu hỏi cốt lõi hơn) để cải thiện khả năng tìm kiếm "
        "trong kho văn bản pháp luật Việt Nam. Chỉ trả lời câu hỏi đã viết lại:"
    ))
    response = get_llm().invoke([rewrite_prompt])
    print(f"  Rewritten: {response.content[:100]}")
    return {"messages": [response]}

## 5. Build the LangGraph

In [ ]:
from langgraph.graph import END, START, StateGraph
from langgraph.prebuilt import ToolNode

workflow = StateGraph(AgentState)

# Nodes
workflow.add_node("agent", agent)
workflow.add_node("retrieve", ToolNode(tools))
workflow.add_node("rewrite", rewrite)
workflow.add_node("generate", generate)

# Edges
workflow.add_edge(START, "agent")
workflow.add_conditional_edges(
    "agent",
    tools_condition,
    {"tools": "retrieve", END: END},
)
workflow.add_conditional_edges("retrieve", grade_documents)
workflow.add_edge("generate", END)
workflow.add_edge("rewrite", "agent")  # retry via agent after rewrite

graph = workflow.compile()
print("Graph compiled.")

In [ ]:
from IPython.display import Image, display

try:
    display(Image(graph.get_graph(xray=True).draw_mermaid_png()))
except Exception as e:
    print(f"Cannot render graph image: {e}")

## 6. Run Queries

Helper that streams each node's output so you can follow the decision path.

In [ ]:
import pprint


def ask(question: str, verbose: bool = True) -> str:
    """Run the agentic RAG graph for a Vietnamese legal question."""
    print(f"\n{'='*70}")
    print(f"Câu hỏi: {question}")
    print('='*70)

    inputs = {"messages": [HumanMessage(content=question)]}
    final_answer = ""

    for output in graph.stream(inputs):
        for node_name, node_output in output.items():
            if verbose:
                print(f"\n[{node_name}]")
            if "messages" in node_output:
                last_msg = node_output["messages"][-1]
                content = getattr(last_msg, "content", str(last_msg))
                if content and node_name == "generate":
                    final_answer = content
                    print("\nTrả lời:\n" + content)
                elif content and verbose and node_name not in ("retrieve",):
                    snippet = content[:200] + ("..." if len(content) > 200 else "")
                    print(f"  {snippet}")

    print()
    return final_answer

### 6.1 Factual query — điều kiện cụ thể

In [ ]:
ask("Điều kiện để thành lập công ty trách nhiệm hữu hạn tại Việt Nam là gì?")

### 6.2 Multi-hop query — sửa đổi / thay thế văn bản

In [ ]:
ask("Luật nào đã sửa đổi bổ sung Luật Doanh nghiệp và nội dung sửa đổi chính là gì?")

### 6.3 Temporal query — hiệu lực theo thời gian

In [ ]:
ask("Các quy định về bảo vệ môi trường trong hoạt động sản xuất còn hiệu lực sau năm 2020 là gì?")

### 6.4 Reasoning query — áp dụng pháp luật vào tình huống

In [ ]:
ask(
    "Một doanh nghiệp xây dựng không có giấy phép xây dựng bị xử phạt như thế nào "
    "theo quy định hiện hành?"
)

## 7. Interactive Chat

Simple REPL for testing queries interactively (stop with `quit` or `exit`).

In [ ]:
while True:
    question = input("\nNhập câu hỏi pháp lý (hoặc 'exit' để thoát): ").strip()
    if question.lower() in ("exit", "quit", ""):
        print("Kết thúc.")
        break
    ask(question)